### Model comparison: XGBoost challenger vs Dixon-Coles

Same inputs as Dixon-Coles (attack/defense from team_ratings) plus
home/away, trained on past seasons and evaluated on the current season —
the same held-out matches Match_Predictions_Builder used for Dixon-Coles.
Metrics only, nothing is written to Gold.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
import xgboost as xgb

PROJECT_ROOT = Path(r"C:\Users\arnol\OneDrive\Documentos\Progaming\Datascience\datascience\football_predictor")
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.parquet_io import read_league_all_seasons

LAKE_ROOT = PROJECT_ROOT / "data"
LEAGUES = ["laliga", "primeira_liga"]
ACTIVE_SEASON = "2026-2027"

silver = {lg: read_league_all_seasons(LAKE_ROOT, layer="silver", league=lg).to_pandas()
          for lg in LEAGUES}
ratings = {lg: read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                       filename="team_ratings.parquet").to_pandas()
           for lg in LEAGUES}
preds = {lg: read_league_all_seasons(LAKE_ROOT, layer="gold", league=lg,
                                     filename="match_predictions.parquet").to_pandas()
         for lg in LEAGUES}

for lg in LEAGUES:
    print(lg, "silver:", silver[lg].shape, "| ratings:", ratings[lg].shape, "| dc preds:", preds[lg].shape)

laliga silver: (1520, 19) | ratings: (2440, 17) | dc preds: (1210, 17)
primeira_liga silver: (1224, 19) | ratings: (1980, 17) | dc preds: (981, 17)


In [2]:
def build_features(matches, rat):
    rt = rat[["season", "matchday", "team_key", "attack", "defense"]]
    m = matches.merge(
        rt.rename(columns={"team_key": "home_team_key", "attack": "h_att", "defense": "h_def"}),
        on=["season", "matchday", "home_team_key"], how="inner")
    m = m.merge(
        rt.rename(columns={"team_key": "away_team_key", "attack": "a_att", "defense": "a_def"}),
        on=["season", "matchday", "away_team_key"], how="inner")
    m["diff_att"] = m["h_att"] - m["a_att"]
    m["diff_def"] = m["h_def"] - m["a_def"]
    return m

feats = {lg: build_features(silver[lg], ratings[lg]) for lg in LEAGUES}
FEATURE_COLS = ["h_att", "h_def", "a_att", "a_def", "diff_att", "diff_def"]

for lg, f in feats.items():
    print(lg, f.shape)

laliga (1220, 25)
primeira_liga (990, 25)


### Train/test split (time-based, no leakage)

Train: seasons 2023-2024 to 2025-2026 (FINISHED matches).
Test: current season (2026-2027), FINISHED — evaluated later against the
same warmup-excluded set used for Dixon-Coles.

In [3]:
TARGET_MAP = {"H": 0, "D": 1, "A": 2}

def split_train_test(f, active_season):
    train = f[(f["season"] != active_season) & (f["status"] == "FINISHED")]
    test = f[(f["season"] == active_season) & (f["status"] == "FINISHED")]
    return train, test

models, splits = {}, {}
for lg in LEAGUES:
    train, test = split_train_test(feats[lg], ACTIVE_SEASON)
    y_train = train["result"].map(TARGET_MAP)

    model = xgb.XGBClassifier(objective="multi:softprob", num_class=3,
                               n_estimators=200, max_depth=3, learning_rate=0.05,
                               eval_metric="mlogloss", random_state=42)
    model.fit(train[FEATURE_COLS], y_train)
    models[lg] = model
    splits[lg] = (train, test)
    print(f"{lg}: train {len(train)} | test {len(test)}")

laliga: train 1140 | test 69
primeira_liga: train 918 | test 62


### Validation: XGBoost vs Dixon-Coles, same evaluated matches

Merge against match_predictions and exclude is_warmup exactly like
Match_Predictions_Builder did, so both models are scored on the
identical match set.

In [4]:
def evaluate(lg):
    model = models[lg]
    _, test = splits[lg]

    dc = preds[lg]
    dc = dc[(dc["season"] == ACTIVE_SEASON) & (dc["result"].notna())]
    ev = test.merge(dc[["season", "matchday", "home_team_key", "away_team_key",
                         "is_warmup", "p_home", "p_draw", "p_away"]],
                     on=["season", "matchday", "home_team_key", "away_team_key"], how="inner")
    ev = ev[~ev["is_warmup"]]

    y = ev["result"].map(TARGET_MAP).to_numpy()
    P_xgb = model.predict_proba(ev[FEATURE_COLS])
    P_dc = ev[["p_home", "p_draw", "p_away"]].to_numpy()

    print(f"=== {lg} === evaluated matches: {len(y)} (warmup excluded)")
    print("XGBoost    -> accuracy:", round(float((P_xgb.argmax(axis=1) == y).mean()), 3),
          "| log-loss:", round(float(log_loss(y, P_xgb, labels=[0, 1, 2])), 4))
    print("Dixon-Coles-> accuracy:", round(float((P_dc.argmax(axis=1) == y).mean()), 3),
          "| log-loss:", round(float(log_loss(y, P_dc, labels=[0, 1, 2])), 4))
    print()

for lg in LEAGUES:
    evaluate(lg)

=== laliga === evaluated matches: 50 (warmup excluded)
XGBoost    -> accuracy: 0.52 | log-loss: 0.9852
Dixon-Coles-> accuracy: 0.5 | log-loss: 0.9855

=== primeira_liga === evaluated matches: 50 (warmup excluded)
XGBoost    -> accuracy: 0.62 | log-loss: 0.892
Dixon-Coles-> accuracy: 0.52 | log-loss: 0.9258



In [5]:
for lg in LEAGUES:
    imp = pd.Series(models[lg].feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print(f"=== {lg} === feature importance")
    print(imp.round(3))
    print()

=== laliga === feature importance
diff_att    0.269
h_att       0.160
a_att       0.160
a_def       0.145
h_def       0.141
diff_def    0.125
dtype: float32

=== primeira_liga === feature importance
diff_att    0.335
diff_def    0.192
a_att       0.129
h_def       0.121
a_def       0.116
h_att       0.108
dtype: float32

